# LoCoMo Dataset Check

`locomo10.json`의 샘플별/세션별 통계를 확인하는 노트북입니다.

이 노트북은 아래 항목을 계산합니다.
- 전체 원소 수
- 각 원소별 세션 수
- 각 원소별 세션당 평균 턴 수
- 각 원소별 세션당 평균 `img_url` 포함 턴 수
- 각 원소별 QA 개수
- 각 원소별 QA `category` 분포
- 각 원소별 QA당 평균 evidence 개수

참고: `conversation` 내부에는 `session_n_date_time`만 있고 실제 `session_n` 리스트가 없는 경우가 있어, 실제 대화 리스트가 존재하는 세션만 집계합니다.

In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

from IPython.display import HTML, display

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None


if pd is not None:
    pd.set_option("display.max_colwidth", None)
    pd.set_option("display.max_columns", None)


def rows_to_html(rows: list[dict], columns: list[str] | None = None) -> HTML:
    if not rows:
        return HTML("<p><i>No rows</i></p>")

    if columns is None:
        columns = list(rows[0].keys())

    header_html = "".join(f"<th>{column}</th>" for column in columns)
    body_html = []
    for row in rows:
        cells = "".join(f"<td>{row.get(column, '')}</td>" for column in columns)
        body_html.append(f"<tr>{cells}</tr>")

    table_html = (
        "<table border='1' style='border-collapse:collapse'>"
        f"<thead><tr>{header_html}</tr></thead>"
        f"<tbody>{''.join(body_html)}</tbody>"
        "</table>"
    )
    return HTML(table_html)


def display_table(rows: list[dict], columns: list[str] | None = None):
    if pd is not None:
        df = pd.DataFrame(rows)
        if columns is not None:
            df = df[columns]
        display(df)
        return df

    display(rows_to_html(rows, columns=columns))
    return rows

In [2]:
DATA_PATH = Path("locomo10.json")

with DATA_PATH.open("r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Loaded {len(data)} samples from {DATA_PATH}")

Loaded 10 samples from locomo10.json


In [3]:
def get_session_keys(conversation: dict) -> list[str]:
    session_keys = [
        key
        for key, value in conversation.items()
        if key.startswith("session_")
        and not key.endswith("_date_time")
        and isinstance(value, list)
    ]
    return sorted(session_keys, key=lambda x: int(x.split("_")[1]))


def count_img_turns(session: list[dict]) -> int:
    return sum(1 for turn in session if isinstance(turn, dict) and turn.get("img_url"))


def summarize_sample(sample: dict) -> tuple[dict, list[dict]]:
    conversation = sample["conversation"]
    qa_list = sample.get("qa", [])
    session_keys = get_session_keys(conversation)
    sessions = [conversation[key] for key in session_keys]

    turn_counts = [len(session) for session in sessions]
    img_turn_counts = [count_img_turns(session) for session in sessions]
    category_counter = Counter(qa.get("category") for qa in qa_list)
    evidence_counts = [len(qa.get("evidence", [])) for qa in qa_list]

    sample_row = {
        "sample_id": sample["sample_id"],
        "num_sessions": len(session_keys),
        "avg_turns_per_session": round(sum(turn_counts) / len(turn_counts), 2) if turn_counts else 0.0,
        "avg_img_turns_per_session": round(sum(img_turn_counts) / len(img_turn_counts), 2) if img_turn_counts else 0.0,
        "num_qa": len(qa_list),
        "avg_evidence_per_qa": round(sum(evidence_counts) / len(evidence_counts), 2) if evidence_counts else 0.0,
        "cat_1": category_counter.get(1, 0),
        "cat_2": category_counter.get(2, 0),
        "cat_3": category_counter.get(3, 0),
        "cat_4": category_counter.get(4, 0),
        "cat_5": category_counter.get(5, 0),
        "category_dist": ", ".join(
            f"cat_{category}: {count}" for category, count in sorted(category_counter.items())
        ),
    }

    session_rows = []
    for session_key, session, turn_count, img_turn_count in zip(
        session_keys, sessions, turn_counts, img_turn_counts
    ):
        session_rows.append(
            {
                "sample_id": sample["sample_id"],
                "session_key": session_key,
                "turn_count": turn_count,
                "img_turn_count": img_turn_count,
                "date_time": conversation.get(f"{session_key}_date_time"),
            }
        )

    return sample_row, session_rows

In [4]:
sample_rows = []
session_rows = []

for sample in data:
    sample_row, per_session_rows = summarize_sample(sample)
    sample_rows.append(sample_row)
    session_rows.extend(per_session_rows)

sample_rows = sorted(sample_rows, key=lambda row: row["sample_id"])
session_rows = sorted(session_rows, key=lambda row: (row["sample_id"], row["session_key"]))

if pd is not None:
    sample_df = pd.DataFrame(sample_rows)
    session_df = pd.DataFrame(session_rows)
else:
    sample_df = sample_rows
    session_df = session_rows

## Overall Summary

In [5]:
overall_summary = [
    {
        "num_samples": len(data),
        "avg_sessions_per_sample": round(sum(row["num_sessions"] for row in sample_rows) / len(sample_rows), 2),
        "avg_turns_per_session_across_samples": round(sum(row["avg_turns_per_session"] for row in sample_rows) / len(sample_rows), 2),
        "avg_img_turns_per_session_across_samples": round(sum(row["avg_img_turns_per_session"] for row in sample_rows) / len(sample_rows), 2),
        "avg_qas_per_sample": round(sum(row["num_qa"] for row in sample_rows) / len(sample_rows), 2),
        "avg_evidence_per_qa_across_samples": round(sum(row["avg_evidence_per_qa"] for row in sample_rows) / len(sample_rows), 2),
    }
]

display_table(overall_summary)

,num_samples,avg_sessions_per_sample,avg_turns_per_session_across_samples,avg_img_turns_per_session_across_samples,avg_qas_per_sample,avg_evidence_per_qa_across_samples
0,10,27.2,21.57,3.33,198.6,1.41


,num_samples,avg_sessions_per_sample,avg_turns_per_session_across_samples,avg_img_turns_per_session_across_samples,avg_qas_per_sample,avg_evidence_per_qa_across_samples
0,10,27.2,21.57,3.33,198.6,1.41


## Sample-Level Table

사용자가 요청한 10개 원소 각각의 통계를 표로 확인합니다.

In [6]:
display_table(sample_rows)

,sample_id,num_sessions,avg_turns_per_session,avg_img_turns_per_session,num_qa,avg_evidence_per_qa,cat_1,cat_2,cat_3,cat_4,cat_5,category_dist
0,conv-26,19,22.05,4.05,199,1.26,32,37,13,70,47,"cat_1: 32, cat_2: 37, cat_3: 13, cat_4: 70, cat_5: 47"
1,conv-30,19,19.42,1.58,105,1.25,11,26,0,44,24,"cat_1: 11, cat_2: 26, cat_4: 44, cat_5: 24"
2,conv-41,32,20.72,2.41,193,1.30,31,27,8,86,41,"cat_1: 31, cat_2: 27, cat_3: 8, cat_4: 86, cat_5: 41"
3,conv-42,29,21.69,2.76,260,1.44,37,40,11,111,61,"cat_1: 37, cat_2: 40, cat_3: 11, cat_4: 111, cat_5: 61"
4,conv-43,29,23.45,4.24,242,1.42,31,26,14,107,64,"cat_1: 31, cat_2: 26, cat_3: 14, cat_4: 107, cat_5: 64"
5,conv-44,28,24.11,5.32,158,1.51,30,24,7,62,35,"cat_1: 30, cat_2: 24, cat_3: 7, cat_4: 62, cat_5: 35"
6,conv-47,31,22.23,2.61,190,1.29,20,34,13,83,40,"cat_1: 20, cat_2: 34, cat_3: 13, cat_4: 83, cat_5: 40"
7,conv-48,30,22.70,3.60,239,1.44,21,42,10,118,48,"cat_1: 21, cat_2: 42, cat_3: 10, cat_4: 118, cat_5: 48"
8,conv-49,25,20.36,3.44,196,1.88,37,33,13,73,40,"cat_1: 37, cat_2: 33, cat_3: 13, cat_4: 73, cat_5: 40"
9,conv-50,30,18.93,3.30,204,1.32,32,32,7,87,46,"cat_1: 32, cat_2: 32, cat_3: 7, cat_4: 87, cat_5: 46"


,sample_id,num_sessions,avg_turns_per_session,avg_img_turns_per_session,num_qa,avg_evidence_per_qa,cat_1,cat_2,cat_3,cat_4,cat_5,category_dist
0,conv-26,19,22.05,4.05,199,1.26,32,37,13,70,47,"cat_1: 32, cat_2: 37, cat_3: 13, cat_4: 70, cat_5: 47"
1,conv-30,19,19.42,1.58,105,1.25,11,26,0,44,24,"cat_1: 11, cat_2: 26, cat_4: 44, cat_5: 24"
2,conv-41,32,20.72,2.41,193,1.30,31,27,8,86,41,"cat_1: 31, cat_2: 27, cat_3: 8, cat_4: 86, cat_5: 41"
3,conv-42,29,21.69,2.76,260,1.44,37,40,11,111,61,"cat_1: 37, cat_2: 40, cat_3: 11, cat_4: 111, cat_5: 61"
4,conv-43,29,23.45,4.24,242,1.42,31,26,14,107,64,"cat_1: 31, cat_2: 26, cat_3: 14, cat_4: 107, cat_5: 64"
5,conv-44,28,24.11,5.32,158,1.51,30,24,7,62,35,"cat_1: 30, cat_2: 24, cat_3: 7, cat_4: 62, cat_5: 35"
6,conv-47,31,22.23,2.61,190,1.29,20,34,13,83,40,"cat_1: 20, cat_2: 34, cat_3: 13, cat_4: 83, cat_5: 40"
7,conv-48,30,22.70,3.60,239,1.44,21,42,10,118,48,"cat_1: 21, cat_2: 42, cat_3: 10, cat_4: 118, cat_5: 48"
8,conv-49,25,20.36,3.44,196,1.88,37,33,13,73,40,"cat_1: 37, cat_2: 33, cat_3: 13, cat_4: 73, cat_5: 40"
9,conv-50,30,18.93,3.30,204,1.32,32,32,7,87,46,"cat_1: 32, cat_2: 32, cat_3: 7, cat_4: 87, cat_5: 46"


## Session-Level Detail

각 원소 내부의 세션별 턴 수와 `img_url` 포함 턴 수를 확인합니다.

In [7]:
display_table(session_rows)

,sample_id,session_key,turn_count,img_turn_count,date_time
0,conv-26,session_1,18,2,"1:56 pm on 8 May, 2023"
1,conv-26,session_10,24,5,"8:56 pm on 20 July, 2023"
2,conv-26,session_11,17,4,"2:24 pm on 14 August, 2023"
3,conv-26,session_12,21,2,"1:50 pm on 17 August, 2023"
4,conv-26,session_13,18,6,"3:31 pm on 23 August, 2023"
...,...,...,...,...,...
267,conv-50,session_5,15,3,"1:16 pm on 3 May, 2023"
268,conv-50,session_6,18,2,"11:50 am on 16 May, 2023"
269,conv-50,session_7,19,2,"6:06 pm on 31 May, 2023"
270,conv-50,session_8,14,2,"2:31 pm on 9 June, 2023"


,sample_id,session_key,turn_count,img_turn_count,date_time
0,conv-26,session_1,18,2,"1:56 pm on 8 May, 2023"
1,conv-26,session_10,24,5,"8:56 pm on 20 July, 2023"
2,conv-26,session_11,17,4,"2:24 pm on 14 August, 2023"
3,conv-26,session_12,21,2,"1:50 pm on 17 August, 2023"
4,conv-26,session_13,18,6,"3:31 pm on 23 August, 2023"
...,...,...,...,...,...
267,conv-50,session_5,15,3,"1:16 pm on 3 May, 2023"
268,conv-50,session_6,18,2,"11:50 am on 16 May, 2023"
269,conv-50,session_7,19,2,"6:06 pm on 31 May, 2023"
270,conv-50,session_8,14,2,"2:31 pm on 9 June, 2023"


## Optional: Per-Sample Session Summary

샘플 하나를 골라 세션별 분포만 보고 싶을 때 사용합니다.

In [8]:
target_sample_id = sample_rows[0]["sample_id"]
display_table([row for row in session_rows if row["sample_id"] == target_sample_id])

,sample_id,session_key,turn_count,img_turn_count,date_time
0,conv-26,session_1,18,2,"1:56 pm on 8 May, 2023"
1,conv-26,session_10,24,5,"8:56 pm on 20 July, 2023"
2,conv-26,session_11,17,4,"2:24 pm on 14 August, 2023"
3,conv-26,session_12,21,2,"1:50 pm on 17 August, 2023"
4,conv-26,session_13,18,6,"3:31 pm on 23 August, 2023"
5,conv-26,session_14,35,10,"1:33 pm on 25 August, 2023"
6,conv-26,session_15,28,2,"3:19 pm on 28 August, 2023"
7,conv-26,session_16,20,6,"12:09 am on 13 September, 2023"
8,conv-26,session_17,26,4,"10:31 am on 13 October, 2023"
9,conv-26,session_18,24,3,"6:55 pm on 20 October, 2023"


,sample_id,session_key,turn_count,img_turn_count,date_time
0,conv-26,session_1,18,2,"1:56 pm on 8 May, 2023"
1,conv-26,session_10,24,5,"8:56 pm on 20 July, 2023"
2,conv-26,session_11,17,4,"2:24 pm on 14 August, 2023"
3,conv-26,session_12,21,2,"1:50 pm on 17 August, 2023"
4,conv-26,session_13,18,6,"3:31 pm on 23 August, 2023"
5,conv-26,session_14,35,10,"1:33 pm on 25 August, 2023"
6,conv-26,session_15,28,2,"3:19 pm on 28 August, 2023"
7,conv-26,session_16,20,6,"12:09 am on 13 September, 2023"
8,conv-26,session_17,26,4,"10:31 am on 13 October, 2023"
9,conv-26,session_18,24,3,"6:55 pm on 20 October, 2023"
